In [1]:
#Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
#Load Core Files
train = pd.read_csv('train.csv', parse_dates=["date"])
stores = pd.read_csv('stores.csv')
oil = pd.read_csv('oil.csv', parse_dates=["date"])
holidays = pd.read_csv('holidays_events.csv', parse_dates=["date"])

In [3]:
#Checks
train.shape
train.head()
train.isnull().sum()

id             0
date           0
store_nbr      0
family         0
sales          0
onpromotion    0
dtype: int64

In [4]:
# Build holiday flags at DATE level (1 row per date) — prevents messy merges

h = holidays.copy()
h["transferred"] = h["transferred"].fillna(False).astype(bool)

# Work Day = makeup workday, not a holiday. Everything else we treat as holiday/event signal.
h["is_holiday_row"] = (h["type"] != "Work Day")

holiday_by_date = (
    h.groupby("date")
     .agg(
         is_holiday=("is_holiday_row", "any"),
         is_transferred=("transferred", "any"),
         has_national=("locale", lambda x: (x == "National").any()),
         has_local=("locale", lambda x: (x == "Local").any()),
         has_regional=("locale", lambda x: (x == "Regional").any()),
         has_bridge=("type", lambda x: (x == "Bridge").any()),
         has_additional=("type", lambda x: (x == "Additional").any()),
         has_event=("type", lambda x: (x == "Event").any()),
     )
     .reset_index()
)

flag_cols = [c for c in holiday_by_date.columns if c != "date"]
holiday_by_date[flag_cols] = holiday_by_date[flag_cols].astype(int)

holiday_by_date.head()

,date,is_holiday,is_transferred,has_national,has_local,has_regional,has_bridge,has_additional,has_event
0,2012-03-02,1,0,0,1,0,0,0,0
1,2012-04-01,1,0,0,0,1,0,0,0
2,2012-04-12,1,0,0,1,0,0,0,0
3,2012-04-14,1,0,0,1,0,0,0,0
4,2012-04-21,1,0,0,1,0,0,0,0


In [5]:
train['sales'].describe()
train['sales'].min()

np.float64(0.0)

In [6]:
#Identify Top Revenue Families
top_families = (
    train.groupby("family")["sales"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

top_families

family
GROCERY I        3.434627e+08
BEVERAGES        2.169545e+08
PRODUCE          1.227047e+08
CLEANING         9.752129e+07
DAIRY            6.448771e+07
BREAD/BAKERY     4.213395e+07
POULTRY          3.187600e+07
MEATS            3.108647e+07
PERSONAL CARE    2.459205e+07
DELI             2.411032e+07
Name: sales, dtype: float64

In [7]:
#We will first create a subset of top 5 Revenue families for ease
top5 = train.groupby("family")["sales"].sum().sort_values(ascending=False).head(5).index
train_top = train[train["family"].isin(top5)].copy()
train_top.shape

(454680, 6)

In [8]:
#Merge Stores
df = train_top.merge(stores, on="store_nbr", how="left")

#Merge oil
df = df.merge(oil, on="date", how="left")
df["dcoilwtico"] = df["dcoilwtico"].ffill().bfill()

#Merge Holidays (FIXED: merge aggregated flags, not raw holidays table)
df = df.merge(holiday_by_date, on="date", how="left")

# Fill missing holiday flags with 0
flag_cols = [c for c in holiday_by_date.columns if c != "date"]
df[flag_cols] = df[flag_cols].fillna(0).astype(int)

# Rename store type column for ease (stores.csv 'type' collides otherwise)
df.rename(columns={"type": "store_type"}, inplace=True)

In [9]:
#Sort Properly
df = df.sort_values(["store_nbr", "family", "date"])

In [10]:
#Save the clean merged dataset
df.to_csv("base_merged_data", index=False)

In [11]:
#Validation Check
df.isnull().sum()
df.head()
df.describe()

,id,date,store_nbr,sales,onpromotion,cluster,dcoilwtico,is_holiday,is_transferred,has_national,has_local,has_regional,has_bridge,has_additional,has_event
count,4.546800e+05,454680,454680.000000,454680.000000,454680.000000,454680.000000,454680.000000,454680.000000,454680.000000,454680.000000,454680.000000,454680.000000,454680.000000,454680.000000,454680.000000
mean,1.500440e+06,2015-04-24 08:27:04.703087616,27.500000,1858.737801,11.721527,8.481481,67.924899,0.147268,0.005344,0.084917,0.062945,0.010689,0.001781,0.022565,0.032660
min,3.000000e+00,2013-01-01 00:00:00,1.000000,0.000000,0.000000,1.000000,26.190000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,7.502235e+05,2014-02-26 18:00:00,14.000000,426.000000,0.000000,4.000000,46.377500,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,1.500444e+06,2015-04-24 12:00:00,27.500000,1122.000000,1.000000,8.500000,53.410000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,2.250664e+06,2016-06-19 06:00:00,41.000000,2507.000000,10.000000,13.000000,95.720000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
max,3.000885e+06,2017-08-15 00:00:00,54.000000,124717.000000,741.000000,17.000000,110.620000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
std,8.662827e+05,NaN,15.585801,2259.023678,27.448766,4.649739,25.669155,0.354374,0.072910,0.278758,0.242865,0.102833,0.042170,0.148513,0.177746


In [12]:
#Fix Oil for 0 null
df["is_holiday"].value_counts(dropna=False)
df["dcoilwtico"].isnull().sum()

np.int64(0)